# Porównanie runów W&B i predykcji modeli

Notebook jest dodatkiem do skryptów:

- `scripts/compare_wandb_runs.py` — porównuje metryki z CSV albo bezpośrednio z W&B API.
- `scripts/compare_checkpoint_predictions.py` — robi side-by-side obraz, ground truth i predykcje wielu checkpointów na tych samych próbkach.

Najpierw wyeksportuj CSV z W&B albo skopiuj plik `wandb_export_*.csv` do katalogu głównego repozytorium / `reports/`.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
CSV_CANDIDATES = [
    *ROOT.glob('wandb_export*.csv'),
    *ROOT.glob('reports/wandb_export*.csv'),
    *Path('.').glob('wandb_export*.csv'),
]

if not CSV_CANDIDATES:
    raise FileNotFoundError('Nie znalazłem wandb_export*.csv. Skopiuj eksport z W&B do repo root albo reports/.')

CSV_PATH = CSV_CANDIDATES[0]
print(CSV_PATH)

df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head(3)


In [ ]:
RUNS = [
    'lraspp_mobilenet_v3_large_segmenter_20260430_011339',
    'lraspp_mobilenet_v3_large_segmenter_20260430_001016',
    'lraspp_mobilenet_v3_large_segmenter_20260430_002026',
    'lraspp_mobilenet_v3_large_segmenter_20260430_001855',
    'advanced_wide_resnet50_2_unet_512_20260610_140127',
    'lraspp_mobilenet_v3_large_segmenter_20260430_003809',
    'lraspp_mobilenet_v3_large_segmenter_first_run',
    'encdecnet_segmenter_20260429_233114',
    'lraspp_mobilenet_v3_large_segmenter_20260430_000633',
    'weighted_augmented_deeplabv3_mobilenet_v3_large_segmenter_20260430_132604',
    'deeplabv3_resnet101_segmenter_20260430_122517',
    'deeplabv3_mobilenet_v3_large_segmenter_20260430_121546',
    'advanced_resnet50_unet_512_20260610_131438',
    'encdecnet_segmenter_20260430_011129',
    'encdecnet_segmenter_20260428_183910',
    'encdecnet_segmenter_20260426_002031',
    'encdecnet_segmenter_20260429_225634',
    'encdecnet_segmenter_20260424_125035',
    'encdecnet_segmenter_20260429_231258',
    'advanced_resnet101_unet_512_20260610_152932',
]

runs = df[df['Name'].isin(RUNS)].copy()
print(len(runs), 'selected runs')
missing = sorted(set(RUNS) - set(runs['Name']))
if missing:
    print('Missing in CSV:')
    for name in missing:
        print('  ', name)


In [ ]:
def family(name: str) -> str:
    if name.startswith('advanced_resnet') or name.startswith('advanced_wide'):
        return 'new/resnet-unet'
    if 'deeplabv3_resnet101' in name:
        return 'old/deeplab-resnet101'
    if 'deeplabv3_mobilenet' in name:
        return 'old/deeplab-mobilenet'
    if 'lraspp' in name:
        return 'old/lraspp'
    if 'encdecnet' in name:
        return 'old/encdecnet'
    return 'other'

runs['family'] = runs['Name'].map(family)
for col in ['val/mean_iou.max', 'val/loss.min', 'val/pixel_acc.max', 'train/mean_iou.max']:
    if col in runs.columns:
        runs[col] = pd.to_numeric(runs[col], errors='coerce')

summary_cols = [
    'Name', 'family', 'State', 'epoch', 'Runtime',
    'val/mean_iou.max', 'val/loss.min', 'val/pixel_acc.max',
    'train/mean_iou.max', 'data_module.batch_size', 'model.backbone.backbone_name',
]
summary_cols = [c for c in summary_cols if c in runs.columns]
summary = runs.sort_values('val/mean_iou.max', ascending=False, na_position='last')[summary_cols]
summary


In [ ]:
OUT = ROOT / 'reports' / 'notebook_wandb_comparison'
OUT.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT / 'selected_run_summary.csv', index=False)
print(OUT / 'selected_run_summary.csv')


In [ ]:
def bar(metric, top_n=20, ascending=False):
    if metric not in runs.columns:
        print(f'Missing metric: {metric}')
        return
    data = runs[['Name', 'family', metric]].dropna().copy()
    data = data.sort_values(metric, ascending=ascending).head(top_n).iloc[::-1]
    fig, ax = plt.subplots(figsize=(12, max(4, 0.45 * len(data))))
    ax.barh(data['Name'], data[metric])
    ax.set_xlabel(metric)
    ax.set_title(metric)
    ax.grid(axis='x', alpha=0.25)
    fig.tight_layout()
    path = OUT / f"bar_{metric.replace('/', '_').replace('.', '_')}.png"
    fig.savefig(path, dpi=180)
    print(path)

bar('val/mean_iou.max', ascending=False)
bar('val/loss.min', ascending=True)
bar('val/pixel_acc.max', ascending=False)


In [ ]:
class_cols = [f'val/iou_class_{i}' for i in range(6) if f'val/iou_class_{i}' in runs.columns]
labels = ['background', 'cap', 'stem', 'gills', 'pores', 'ring'][:len(class_cols)]

if class_cols:
    data = runs[['Name', 'val/mean_iou.max', *class_cols]].copy()
    for c in ['val/mean_iou.max', *class_cols]:
        data[c] = pd.to_numeric(data[c], errors='coerce')
    data = data.sort_values('val/mean_iou.max', ascending=False, na_position='last').head(8)

    x = np.arange(len(class_cols))
    width = 0.8 / max(1, len(data))
    fig, ax = plt.subplots(figsize=(14, 6))
    for i, (_, row) in enumerate(data.iterrows()):
        ax.bar(x + (i - (len(data)-1)/2) * width, [row[c] for c in class_cols], width=width, label=row['Name'])
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1)
    ax.set_ylabel('IoU')
    ax.set_title('Per-class validation IoU')
    ax.grid(axis='y', alpha=0.25)
    ax.legend(fontsize=8)
    fig.tight_layout()
    path = OUT / 'per_class_val_iou.png'
    fig.savefig(path, dpi=180)
    print(path)
else:
    print('CSV nie ma kolumn val/iou_class_*')


In [ ]:
# Grupowe porównanie rodzin modeli.
metric = 'val/mean_iou.max'
if metric in runs.columns:
    group = runs.groupby('family')[metric].agg(['count', 'max', 'mean', 'median']).sort_values('max', ascending=False)
    display(group)


## Automatyczny raport z CLI

Ten sam wynik bez notebooka:


In [ ]:
cmd = f'''uv run python -m scripts.compare_wandb_runs \
  --csv {CSV_PATH} \
  --run {','.join(RUNS)} \
  --primary_metric val/mean_iou.max \
  --output_dir reports/wandb_comparison'''
print(cmd)


## Porównanie predykcji na tych samych obrazach

Wpisz ścieżki do checkpointów. Pierwszy model jest bazowy: z niego wybierane są filename'y. Skrypt zapisze `selected_filenames.txt`, więc potem możesz odpalać kolejne porównania dokładnie na tych samych przykładach.


In [ ]:
PREDICTION_COMPARE_CMD = r'''
uv run python -m scripts.compare_checkpoint_predictions \
  --model resnet50=src/config/resnet50_unet_512.py:logs/advanced_resnet50_unet_512_20260610_131438/last.ckpt \
  --model resnet101=src/config/resnet101_unet_512.py:logs/advanced_resnet101_unet_512_20260610_152932/last.ckpt \
  --split val \
  --prefer_class 4 \
  --prefer_class 5 \
  --num_samples 12 \
  --output_dir reports/prediction_comparison
'''
print(PREDICTION_COMPARE_CMD)


Jeżeli chcesz porównać z konkretnymi starymi modelami, dodaj kolejne `--model`, np. checkpoint LRASPP albo DeepLabV3. Ważne: config i checkpoint muszą pasować do siebie.
